# Chapter 2 — Working with text data

Use this notebook to follow along with Chapter 2. Add your notes, experiments, and implementations below as you work through the chapter.

## 2.1 Reading a text corpus

Language models learn from text corpora. This chapter uses Edith Wharton's short story *The Verdict* as a small, manageable dataset.

The next cell downloads the UTF-8 text file from the book's companion repository and saves it beside this notebook. Once downloaded, the local copy can be reused without another network request.

In [1]:
# Download the chapter's sample corpus to the notebook directory.
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x22bf52fbce0>)

### Inspecting the raw text

Before tokenizing, load the entire file into one string and inspect its size and opening characters. Looking at a sample helps catch encoding, path, or formatting problems early.

`raw_text` is the source sequence from which the tokenizer will build discrete tokens.

In [2]:
# Read the complete UTF-8 corpus as one string.
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Inspect the corpus size and a short sample before preprocessing.
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2.2 Tokenizing text

A tokenizer converts a string into smaller units called **tokens**. Tokens may be words, punctuation marks, subwords, or special symbols.

We begin with a deliberately simple regular-expression tokenizer. The capturing group in `r'(\s)'` keeps matched whitespace in the result, making each split visible while we develop the rule.

In [3]:
import re

# Start by splitting on whitespace while retaining the separators.
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


### Keeping punctuation as tokens

Punctuation carries useful structure, so it should be separated rather than discarded. Adding comma and period to the capturing group returns those delimiters as individual list items.

The split also creates empty strings where delimiters touch. We remove those artifacts in the following step.

In [4]:
# Capturing commas, periods, and whitespace keeps each delimiter in the result.
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


### Removing whitespace and empty fragments

The list comprehension keeps only items whose stripped value is non-empty. This removes separator whitespace and empty strings while preserving words and punctuation.

In [5]:
# Remove whitespace-only and empty fragments created by re.split.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


### Expanding the tokenization rule

Real text contains more than commas and periods. The next pattern handles several punctuation marks plus a double hyphen (`--`), then normalizes the result by stripping and filtering each fragment.

This is still a handcrafted tokenizer: it is useful for understanding the mechanics, but production tokenizers need broader rules and a strategy for unseen text.

In [6]:
# Expand the tokenizer to recognize common punctuation and double hyphens.
text = "Hello, world. Is this-- a test?"
result = re.split(r"([,.:;?_!\"()']|--|\s)", text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


## 2.3 Tokenizing the complete corpus

Now apply the same rule to the full story. The resulting `preprocessed` list is the ordered token sequence that can later be converted into token IDs.

Printing its length gives the corpus size under this particular tokenization scheme. A different tokenizer can produce a different token count for the same text.

In [7]:
# Apply the same tokenization rule to the complete corpus.
preprocessed = re.split(r"([,.:;?_!\"()']|--|\s)", raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


### Inspecting the token sequence

A short sample verifies that words and punctuation were separated as intended before vocabulary construction.

In [8]:
# Inspect the opening tokens to verify the preprocessing result.
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.4 Building a vocabulary

A **vocabulary** contains every distinct token the tokenizer knows. Converting `preprocessed` to a set removes duplicates; sorting makes the token-to-ID mapping deterministic and easier to inspect.

`vocab_size` is the number of unique tokens in this corpus, not the total number of token occurrences.

In [9]:
# A sorted set yields a deterministic list of unique corpus tokens.
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


### Mapping tokens to integer IDs

Neural networks operate on numbers rather than strings. `enumerate` assigns one integer ID to each token, producing the lookup dictionary `vocab`.

The sample output shows that punctuation and words occupy the same vocabulary and each receive their own ID.

In [10]:
# Assign a unique integer ID to every token.
vocab = {token:integer for integer,token in enumerate(all_words)}

# Display only the beginning of the vocabulary.
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


## 2.5 A simple text tokenizer

`SimpleTokenizerV1` stores mappings in both directions:

- `str_to_int` supports **encoding** text as token IDs.
- `int_to_str` supports **decoding** IDs back into readable text.

Encoding repeats the preprocessing rule used to build the vocabulary. Decoding joins the tokens and then removes spaces that were inserted before punctuation.

In [11]:
class SimpleTokenizerV1:
    """Encode known text tokens as IDs and decode IDs back to text."""

    def __init__(self, vocab: dict[str, int]) -> None:
        """Create forward and reverse lookup tables from a vocabulary."""
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        """Tokenize text and return the corresponding vocabulary IDs."""
        preprocessed = re.split(r"([,.?_!\"()']|--|\s)", text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        """Convert token IDs to text and restore punctuation spacing."""
        text = " ".join([self.int_to_str[i] for i in ids])

        # Remove spaces inserted immediately before punctuation.
        text = re.sub(r"\s+([,.?!\"()'])", r"\1", text)
        return text

### Encoding and decoding an in-vocabulary passage

This example uses text from the training corpus, so every token should exist in `vocab`. The encoded list is the numerical representation a language model would consume.

In [12]:
# Version 1 can encode passages containing only known vocabulary tokens.
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [13]:
# Decode the IDs to check that the text can be reconstructed.
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


### The out-of-vocabulary problem

A word that was absent from the corpus has no entry in `vocab`. Encoding such text with version 1 raises a `KeyError`. The commented call demonstrates this limitation without interrupting the notebook.

In [14]:
# "Hello" is absent from this corpus vocabulary, so version 1 raises KeyError.
text = "Hello, do you like tea?"
# print(tokenizer.encode(text)) # KeyError

## 2.6 Adding special context tokens

Special tokens let a tokenizer represent situations that ordinary corpus tokens cannot:

- `<|unk|>` replaces an unknown, out-of-vocabulary token.
- `<|endoftext|>` marks a document boundary, allowing multiple independent texts to be combined.

They are appended before rebuilding the vocabulary so they receive integer IDs like every other token.

In [15]:
# Add markers for document boundaries and unknown tokens.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


In [16]:
# Confirm that the special tokens were appended to the vocabulary.
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Handling unknown tokens

`SimpleTokenizerV2` checks each parsed token during encoding and substitutes `<|unk|>` whenever the token is missing from the vocabulary. This makes encoding robust to new text while preserving the same decoding process.

In [17]:
class SimpleTokenizerV2:
    """A word-level tokenizer that replaces unseen tokens with <|unk|>."""

    def __init__(self, vocab: dict[str, int]) -> None:
        """Create forward and reverse lookup tables from a vocabulary."""
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        """Tokenize text, map unknown tokens, and return token IDs."""
        preprocessed = re.split(r"([,.:;?_!\"()']|--|\s)", text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Substitute the unknown-token marker before looking up IDs.
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        """Convert token IDs to text and restore punctuation spacing."""
        text = " ".join([self.int_to_str[i] for i in ids])

        # Remove spaces inserted immediately before punctuation.
        text = re.sub(r"\s+([,.:;?!\"()'])", r"\1", text)
        return text

### Joining independent documents

The end-of-text token separates passages that are not naturally continuous. The next cells verify that version 2 can encode new words as `<|unk|>`, retain the document boundary, and decode the complete sequence.

In [18]:
# Mark the boundary between two independent text samples.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [19]:
# Unknown words are mapped to <|unk|> instead of raising an error.
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [20]:
# Round-trip the joined documents through the tokenizer.
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 2.7 Byte pair encoding

The handcrafted tokenizers clarify the core ideas, but their fixed word vocabulary does not scale well. **Byte pair encoding (BPE)** builds a vocabulary of frequently occurring subword units. Common words can remain single tokens, while rare or invented words are decomposed into smaller pieces.

### Using `tiktoken`

`tiktoken` provides OpenAI's production-grade byte pair encoding implementations. The version check makes the environment explicit and helps reproduce the notebook.

In [21]:
# Report the installed tokenizer version for reproducibility.
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


### Loading the GPT-2 encoding

Unlike the word-level tokenizer above, GPT-2's BPE tokenizer can represent unfamiliar words by splitting them into known byte-based subword units. This avoids a dedicated unknown-word token for ordinary input.

In [22]:
# Load the byte pair encoding vocabulary used by GPT-2.
tokenizer = tiktoken.get_encoding("gpt2")

### Encoding text with BPE

`allowed_special` explicitly permits GPT-2's end-of-text marker. Notice that the example includes an invented place name: BPE can still encode it as a sequence of smaller known units.

In [23]:
# BPE can represent unfamiliar words as sequences of subword tokens.
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [24]:
# Decode the BPE IDs to reconstruct the original string.
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## Exercise 2.1 — Byte pair encoding of unknown words

Encode an unfamiliar phrase and inspect both its token IDs and decoded pieces. The result demonstrates how BPE represents text that never appeared as a complete vocabulary entry.

In [25]:
# Exercise: inspect how GPT-2 tokenizes and reconstructs an unusual phrase.
weird_phrase = "Akwirw ier"
tokens = tokenizer.encode(weird_phrase, allowed_special={"<|endoftext|>"})
print(tokens)
strings = tokenizer.decode(tokens)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


## Chapter checkpoint

This notebook has now followed the text-processing pipeline from raw characters to model-ready token IDs:

1. load and inspect a text corpus;
2. split text into word and punctuation tokens;
3. build a vocabulary and token-to-ID mapping;
4. implement encoding and decoding;
5. handle unknown words and document boundaries with special tokens; and
6. use GPT-2 byte pair encoding to represent arbitrary text as subword tokens.

These integer token sequences are the inputs used to create embeddings and training examples in the next stages of the language-model pipeline.

Data sampling with a sliding window

In [26]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


Next, we remove the first 50 tokens from the dataset for demonstration purposes, as it results in a slightly more interesting text passage in the next steps:



In [27]:
enc_sample = enc_text[50:]


In [28]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [29]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [30]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [31]:
import torch
import tiktoken
from torch.utils.data import DataLoader, Dataset


class GPTDatasetV1(Dataset):
    """Create overlapping input-target token windows from a text corpus."""

    def __init__(
        self,
        txt: str,
        tokenizer: tiktoken.Encoding,
        max_length: int,
        stride: int,
    ) -> None:
        """Tokenize text and materialize shifted windows as tensors."""
        # Empty collections need annotations because their item type is not inferable.
        self.input_ids: list[torch.Tensor] = []
        self.target_ids: list[torch.Tensor] = []

        token_ids = tokenizer.encode(txt)

        # Stride controls the overlap between adjacent training examples.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self) -> int:
        """Return the number of available input-target windows."""
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Return one input window and its one-token-shifted target window."""
        return self.input_ids[idx], self.target_ids[idx]

C:\Users\giloz\dev\build-llms-from-scratch-companion\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
